In [2]:
"""
Step 4: Split dpo_pairs.jsonl into training and validation sets.

Stratifies by:
  - meta_source (real vs synthetic) — so validation isn't accidentally all-real or all-synthetic
  - relevant vs irrelevant — so validation reflects your true class balance
  - meta_trap_type (for irrelevant pairs) — so validation covers a spread of near-miss
    categories (wrong source, wrong property focus, additive-only, etc.) rather than
    only the most common trap type

Requires the pairs to have been built with the updated step3_build_dpo_pairs.py
(the one that tags meta_source and meta_trap_type). If your current dpo_pairs.jsonl
predates that update, rerun step3 first, then run this script.

Output:
  train_pairs.jsonl       (~85% of pairs, used for DPO fine-tuning and as the
                            source pool for few-shot exemplar selection)
  validation_pairs.jsonl  (~15% of pairs, held out — never used in training or
                            as few-shot exemplars, only for evaluation)
"""

import json
import random
from collections import defaultdict

VALIDATION_FRACTION = 0.15
RANDOM_SEED = 42  # fixed seed so the split is reproducible if you rerun this


def load_pairs(path):
    with open(path) as f:
        return [json.loads(line) for line in f]


def stratify_key(pair):
    """Group pairs into strata for balanced sampling.
    Relevant pairs: one stratum ('relevant', 'real') or ('relevant', 'synthetic').
    Irrelevant pairs: stratum by (source, trap_type), so each trap type gets
    representation in both train and validation."""
    is_relevant = pair["chosen"].strip().lower().startswith("relevant")
    source = pair.get("meta_source", "real")  # fallback for older files without metadata
    if is_relevant:
        return ("relevant", source)
    else:
        trap = pair.get("meta_trap_type") or "unspecified"
        return ("irrelevant", source, trap)


def stratified_split(pairs, validation_fraction, seed):
    random.seed(seed)
    strata = defaultdict(list)
    for p in pairs:
        strata[stratify_key(p)].append(p)

    train, validation = [], []
    for key, group in strata.items():
        random.shuffle(group)
        n_val = max(1, round(len(group) * validation_fraction)) if len(group) >= 3 else 0
        validation.extend(group[:n_val])
        train.extend(group[n_val:])

    random.shuffle(train)
    random.shuffle(validation)
    return train, validation


def summarize(pairs, label):
    print(f"\n{label}: {len(pairs)} pairs")
    counts = defaultdict(int)
    for p in pairs:
        counts[stratify_key(p)] += 1
    for key in sorted(counts, key=str):
        print(f"  {key}: {counts[key]}")


def main():
    pairs = load_pairs("dpo_pairs.jsonl")

    has_metadata = all("meta_source" in p for p in pairs)
    if not has_metadata:
        print(
            "WARNING: dpo_pairs.jsonl does not have meta_source/meta_trap_type tags.\n"
            "This means real vs synthetic and trap-type stratification will not work,\n"
            "the split will fall back to a plain relevant/irrelevant stratified split.\n"
            "Rerun the updated step3_build_dpo_pairs.py to regenerate pairs with metadata\n"
            "for a fully stratified split.\n"
        )

    train, validation = stratified_split(pairs, VALIDATION_FRACTION, RANDOM_SEED)

    summarize(train, "TRAIN")
    summarize(validation, "VALIDATION")

    with open("train_pairs.jsonl", "w") as f:
        for p in train:
            f.write(json.dumps(p) + "\n")

    with open("validation_pairs.jsonl", "w") as f:
        for p in validation:
            f.write(json.dumps(p) + "\n")

    print(f"\nTotal: {len(pairs)} pairs -> {len(train)} train / {len(validation)} validation")
    print("Saved to train_pairs.jsonl and validation_pairs.jsonl")
    print(
        "\nReminder: validation_pairs.jsonl must NOT be used for DPO fine-tuning, "
        "and must NOT be used as few-shot exemplars for the GPT/Gemini comparison. "
        "Few-shot exemplars should be selected only from train_pairs.jsonl."
    )


if __name__ == "__main__":
    main()


TRAIN: 369 pairs
  ('irrelevant', 'real', 'algae_source'): 2
  ('irrelevant', 'real', 'animal_source'): 5
  ('irrelevant', 'real', 'fish_or_seafood_source'): 10
  ('irrelevant', 'real', 'fungal_source'): 2
  ('irrelevant', 'real', 'insect_source'): 2
  ('irrelevant', 'real', 'microbial_source'): 2
  ('irrelevant', 'real', 'non_protein_material'): 16
  ('irrelevant', 'real', 'poultry_source'): 2
  ('irrelevant', 'real', 'unspecified'): 2
  ('irrelevant', 'real', 'wrong_property_focus'): 6
  ('irrelevant', 'synthetic', 'algae_source'): 12
  ('irrelevant', 'synthetic', 'animal_source'): 13
  ('irrelevant', 'synthetic', 'fish_or_seafood_source'): 13
  ('irrelevant', 'synthetic', 'fungal_source'): 13
  ('irrelevant', 'synthetic', 'insect_source'): 13
  ('irrelevant', 'synthetic', 'microbial_source'): 13
  ('irrelevant', 'synthetic', 'non_protein_material'): 13
  ('irrelevant', 'synthetic', 'poultry_source'): 13
  ('irrelevant', 'synthetic', 'wrong_property_focus'): 12
  ('relevant', 'real'